In [45]:
import pandas as pd
raw_application_df = pd.read_csv('raw_data/IS453 Group Assignment - Application Data.csv')
raw_bureau_df = pd.read_csv('raw_data/IS453 Group Assignment - Bureau Data.csv')

In [46]:
raw_bureau_df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372202,215474,5554621,Active,currency 1,-2891,0,-1766.0,NaN,0.0,0,225000.0,0.0,0.0,0.0,Credit card,-549,NaN
372203,215474,5554624,Closed,currency 1,-2263,0,NaN,-704.0,0.0,0,0.0,0.0,0.0,0.0,Credit card,-704,NaN
372204,440726,5554632,Closed,currency 1,-870,0,-811.0,-811.0,NaN,0,44469.0,0.0,0.0,0.0,Consumer credit,-810,NaN
372205,305396,5554650,Closed,currency 1,-1745,0,-1533.0,-1592.0,0.0,0,41972.4,0.0,0.0,0.0,Consumer credit,-1592,NaN


In [47]:
#FILTERING IN APPLICATION DATA

#filter out 'FLAG_OWN_REALTY'] == 'Y'
cleaned_application_df = raw_application_df[raw_application_df['FLAG_OWN_REALTY'] != 'Y']

#filter out people who already own a house
allowed_types = ['Rented apartment', 'With parents', 'Municipal apartment', 'Office apartment']
cleaned_application_df = cleaned_application_df[cleaned_application_df['NAME_HOUSING_TYPE'].isin(allowed_types)]

#filter out people under 21
cleaned_application_df = cleaned_application_df[cleaned_application_df['DAYS_BIRTH'] <= -7665]

# # keep only those rows without missing goods price
# cleaned_application_df = cleaned_application_df[cleaned_application_df['AMT_GOODS_PRICE'].notna()]

In [19]:
# refer to: 100002

# CREDIT_ACTIVE flatten to count of TOTAL_ACTIVE and count of TOTAL_CLOSED etc. - DIRTI

# CREDIT_CURRENCY:(Assumption: currency 1 is local) -> currency 2,3,4 falls into foreign currency category and we split by counts for those columns - DIRTI
raw_bureau_df['CREDIT_CURRENCY'].value_counts()

# DAYS_CREDIT - ANSHIKA
# DAYS_CREDIT_RECENT- How recently the client borrowed (min)
# DAYS_CREDIT_OLDEST - How long client has been borrowing (max)
# RECENT_LOAN_COUNT - number of active loans within 100 days

# CREDIT_DAY_OVERDUE take average of rows to AVERAGE_CREDIT_DAY_OVERDUE  - ANSHIKA

# DAYS_CREDIT_ENDDATE take average of rows that are negative values and ACTIVE then create AVERAGE_DAYS_OVERDUE if all are closed value = 0 - ANSHIKA

# DAYS_ENDDATE_FACT - ROOPA
# 1. Loan durations
# LOAN_DURATION_SCHEDULED = DAYS_CREDIT_ENDDATE - DAYS_CREDIT
# LOAN_DURATION_ACTUAL = DAYS_ENDDATE_FACT - DAYS_CREDIT
# 2. Loan Deviation
# LOAN_DEVIATION = DAYS_ENDDATE_FACT - DAYS_CREDIT_ENDDATE: Positive → late repayment,Negative → early repayment
# MAX_RECENCY_CLOSED_LOAN	Most recent closed loan
# AVG_LOAN_DURATION_ACTUAL	Average duration of closed loans

# AMT_CREDIT_MAX_OVERDUE - ROOPA
# keep MAX_OVERDUE_AMOUNT: Max overdue = 5043.645
# COUNT_OVERDUE_LOANS: Loans with overdue > 0 = 3

# CNT_CREDIT_PROLONG: SUM_CNT_CREDIT_PROLONG - JEWEL
# - sum up how many extensions the guy asked for

# AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT - SHINU
# - AMT_CREDIT_SUM [sum up all the rows for 1 guy] divide by AMT_CREDIT_SUM_DEBT [sum up all the rows for 1 guy] -> TOTAL_DEBT_RATIO

# AMT_CREDIT_SUM_LIMIT - JEWEL
# - TOTAL_CREDIT_LIMIT: Sum up all of the AMT_CREDIT_SUM_LIMIT

# AMT_CREDIT_SUM_OVERDUE - SHINU
# - find out the maximum of the AMT_CREDIT_SUM_OVERDUE for the active loans for the guy -> MAX_ACTIVE_AMT_CREDIT_SUM_OVERDUE -> 0 if all closed

# CREDIT_TYPE to count of TOTAL of the different types of credits. - JEWEL

# DAYS_CREDIT_UPDATE - just average all the rows and get a AVERAGE_DAYS_CREDIT_UPDATE  - NORA

# AMT_ANNUITY - NORA
# - TOTAL_ANNUITY → sum of active loan's annuities → total monthly obligations


CREDIT_CURRENCY
currency 1    371883
currency 2       286
currency 3        35
currency 4         3
Name: count, dtype: int64

In [20]:
#SHINU - AMT_CREDIT_SUM_OVERDUE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,CREDIT_DAY_OVERDUE
import numpy as np

df = raw_bureau_df.copy()

g = df.groupby('SK_ID_CURR')

# max overdue on Active loans; no Active loan -> 0
amt_overdue_client = (
    df.loc[df['CREDIT_ACTIVE'] == 'Active']
    .groupby('SK_ID_CURR')['AMT_CREDIT_SUM_OVERDUE']
    .max()
)

# Map to new column
df['MAX_AMT_CREDIT_SUM_OVERDUE'] = (
    df['SK_ID_CURR']
    .map(amt_overdue_client)
    .fillna(0)
)


# Ratio uses sums per person; same value on every row for that client
sum_credit = g['AMT_CREDIT_SUM'].sum()
sum_debt = g['AMT_CREDIT_SUM_DEBT'].sum()
# total_debt_ratio = sum_credit.div(sum_debt.replace(0, np.nan))
total_debt_ratio = sum_debt.div(sum_credit.replace(0, np.nan))
df['DEBT_RATIO'] = df['SK_ID_CURR'].map(total_debt_ratio)

#taking average overdue days per person
avg_overdue = g['CREDIT_DAY_OVERDUE'].mean()
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(avg_overdue)
df[df['SK_ID_CURR']==100002]



,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MAX_AMT_CREDIT_SUM_OVERDUE,DEBT_RATIO,AVERAGE_CREDIT_DAY_OVERDUE


In [21]:
# JEWEL

# AMT_CREDIT_SUM_LIMIT
# - TOTAL_CREDIT_LIMIT: Sum up all of the AMT_CREDIT_SUM_LIMIT
df[df['SK_ID_CURR']==100002]
g = df.groupby('SK_ID_CURR')
total_limit = g['AMT_CREDIT_SUM_LIMIT'].sum()
df['TOTAL_CREDIT_LIMIT'] = df['SK_ID_CURR'].map(total_limit)
df[df['SK_ID_CURR']==100002]

# CNT_CREDIT_PROLONG: SUM_CNT_CREDIT_PROLONG
# - sum up how many extensions the guy asked for
g = df.groupby('SK_ID_CURR')
sum_prolong = g['CNT_CREDIT_PROLONG'].sum()
df['SUM_CNT_CREDIT_PROLONG'] = df['SK_ID_CURR'].map(sum_prolong)
df[df['SK_ID_CURR']==100002]

# CREDIT_TYPE to count of TOTAL of the different types of credits.
# Create dummies and attach directly to df
df = pd.concat([df, pd.get_dummies(df['CREDIT_TYPE'], prefix='CREDIT_TYPE')], axis=1)

# Only dummy columns (exclude original CREDIT_TYPE)
credit_type_cols = [col for col in df.columns
                    if col.startswith('CREDIT_TYPE_') and col != 'CREDIT_TYPE']

# Groupby
g = df.groupby('SK_ID_CURR')

# Map counts back
for col in credit_type_cols:
    df[col] = df['SK_ID_CURR'].map(g[col].sum())

df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,CREDIT_TYPE_Consumer credit,CREDIT_TYPE_Credit card,CREDIT_TYPE_Loan for business development,CREDIT_TYPE_Loan for the purchase of equipment,CREDIT_TYPE_Loan for working capital replenishment,CREDIT_TYPE_Microloan,CREDIT_TYPE_Mobile operator loan,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,7,3,0,0,0,0,0,0,0,0
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,7,3,0,0,0,0,0,0,0,0
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,7,3,0,0,0,0,0,0,0,0
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,7,3,0,0,0,0,0,0,0,0
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,7,3,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372202,215474,5554621,Active,currency 1,-2891,0,-1766.0,NaN,0.0,0,...,0,2,0,0,0,0,0,0,0,0
372203,215474,5554624,Closed,currency 1,-2263,0,NaN,-704.0,0.0,0,...,0,2,0,0,0,0,0,0,0,0
372204,440726,5554632,Closed,currency 1,-870,0,-811.0,-811.0,NaN,0,...,1,0,0,0,0,0,0,0,0,0
372205,305396,5554650,Closed,currency 1,-1745,0,-1533.0,-1592.0,0.0,0,...,1,1,0,0,0,0,0,0,0,0


In [22]:
#roopa

# DAYS_ENDDATE_FACT - ROOPA
# 1. Loan durations
# LOAN_DURATION_SCHEDULED = DAYS_CREDIT_ENDDATE - DAYS_CREDIT
# LOAN_DURATION_ACTUAL = DAYS_ENDDATE_FACT - DAYS_CREDIT
# 2. Loan Deviation
# LOAN_DEVIATION = DAYS_ENDDATE_FACT - DAYS_CREDIT_ENDDATE: Positive → late repayment,Negative → early repayment
# MAX_RECENCY_CLOSED_LOAN	Most recent closed loan
# AVG_LOAN_DURATION_ACTUAL	Average duration of closed loans

# df = raw_bureau_df.copy()

df["LOAN_DURATION_SCHEDULED"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_CREDIT"]
df["LOAN_DURATION_ACTUAL"] = df["DAYS_ENDDATE_FACT"] - df["DAYS_CREDIT"]
df["LOAN_DEVIATION"] = df["DAYS_CREDIT_ENDDATE"] - df["DAYS_ENDDATE_FACT"]

# Use only closed loans for these metrics
closed_loans = df[df["CREDIT_ACTIVE"] == "Closed"].copy()
closed_group = closed_loans.groupby("SK_ID_CURR")

# User-level aggregates (closed loans)
max_recency_closed = closed_group["DAYS_ENDDATE_FACT"].max()
avg_duration_actual = closed_group["LOAN_DURATION_ACTUAL"].mean()

# Map DAYS_ENDDATE_FACT features back to bureau rows
df["MAX_RECENCY_CLOSED_LOAN"] = df["SK_ID_CURR"].map(max_recency_closed)
df["AVG_LOAN_DURATION_ACTUAL"] = df["SK_ID_CURR"].map(avg_duration_actual)

# AMT_CREDIT_MAX_OVERDUE features
overdue_group = df.assign(
    AMT_CREDIT_MAX_OVERDUE=df["AMT_CREDIT_MAX_OVERDUE"].fillna(0)
).groupby("SK_ID_CURR")

max_overdue_amount = overdue_group["AMT_CREDIT_MAX_OVERDUE"].max()
count_overdue_loans = overdue_group["AMT_CREDIT_MAX_OVERDUE"].apply(lambda s: (s > 0).sum())

# Map overdue features back to bureau rows
df["MAX_OVERDUE_AMOUNT"] = df["SK_ID_CURR"].map(max_overdue_amount)
df["COUNT_OVERDUE_LOANS"] = df["SK_ID_CURR"].map(count_overdue_loans)
df

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,CREDIT_TYPE_Mortgage,CREDIT_TYPE_Real estate loan,CREDIT_TYPE_Unknown type of loan,LOAN_DURATION_SCHEDULED,LOAN_DURATION_ACTUAL,LOAN_DEVIATION,MAX_RECENCY_CLOSED_LOAN,AVG_LOAN_DURATION_ACTUAL,MAX_OVERDUE_AMOUNT,COUNT_OVERDUE_LOANS
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,0,0,0,344.0,344.0,0.0,-153.0,399.6,77674.5,1
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,0,0,0,1283.0,NaN,NaN,-153.0,399.6,77674.5,1
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,0,0,0,731.0,NaN,NaN,-153.0,399.6,77674.5,1
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,0,0,0,NaN,NaN,NaN,-153.0,399.6,77674.5,1
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,0,0,0,1826.0,NaN,NaN,-153.0,399.6,77674.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372202,215474,5554621,Active,currency 1,-2891,0,-1766.0,NaN,0.0,0,...,0,0,0,1125.0,NaN,NaN,-704.0,1559.0,0.0,0
372203,215474,5554624,Closed,currency 1,-2263,0,NaN,-704.0,0.0,0,...,0,0,0,NaN,1559.0,NaN,-704.0,1559.0,0.0,0
372204,440726,5554632,Closed,currency 1,-870,0,-811.0,-811.0,NaN,0,...,0,0,0,59.0,59.0,0.0,-811.0,59.0,0.0,0
372205,305396,5554650,Closed,currency 1,-1745,0,-1533.0,-1592.0,0.0,0,...,0,0,0,212.0,153.0,59.0,-624.0,114.5,0.0,0


In [23]:
# Driti - CREDIT_ACTIVE, CREDIT_CURRENCY

# df = raw_bureau_df.copy()
g = df.groupby('SK_ID_CURR')

# CREDIT_ACTIVE: count of each status per client
active_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Active').sum())
closed_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Closed').sum())
sold_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Sold').sum())
bad_debt_count = g['CREDIT_ACTIVE'].apply(lambda x: (x == 'Bad debt').sum())

df['TOTAL_ACTIVE'] = df['SK_ID_CURR'].map(active_count)
df['TOTAL_CLOSED'] = df['SK_ID_CURR'].map(closed_count)
df['TOTAL_SOLD'] = df['SK_ID_CURR'].map(sold_count)
df['TOTAL_BAD_DEBT'] = df['SK_ID_CURR'].map(bad_debt_count)

# CREDIT_CURRENCY: local vs foreign loan counts
# Assumption: currency 1 is local, everything else is foreign
local_count = g['CREDIT_CURRENCY'].apply(lambda x: (x == 'currency 1').sum())
foreign_count = g['CREDIT_CURRENCY'].apply(lambda x: (x != 'currency 1').sum())

df['COUNT_LOCAL_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(local_count)
df['COUNT_FOREIGN_CURRENCY_LOANS'] = df['SK_ID_CURR'].map(foreign_count)

# Verify with one customer
df[df['SK_ID_CURR']==215354][['SK_ID_CURR', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
    'TOTAL_ACTIVE', 'TOTAL_CLOSED', 'TOTAL_SOLD', 'TOTAL_BAD_DEBT',
    'COUNT_LOCAL_CURRENCY_LOANS', 'COUNT_FOREIGN_CURRENCY_LOANS']]

,SK_ID_CURR,CREDIT_ACTIVE,CREDIT_CURRENCY,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS
0,215354,Closed,currency 1,6,5,0,0,11,0
1,215354,Active,currency 1,6,5,0,0,11,0
2,215354,Active,currency 1,6,5,0,0,11,0
3,215354,Active,currency 1,6,5,0,0,11,0
4,215354,Active,currency 1,6,5,0,0,11,0
5,215354,Active,currency 1,6,5,0,0,11,0
6,215354,Active,currency 1,6,5,0,0,11,0
225157,215354,Closed,currency 1,6,5,0,0,11,0
225158,215354,Closed,currency 1,6,5,0,0,11,0
225159,215354,Closed,currency 1,6,5,0,0,11,0


In [24]:
df[df['SK_ID_CURR']==100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,MAX_RECENCY_CLOSED_LOAN,AVG_LOAN_DURATION_ACTUAL,MAX_OVERDUE_AMOUNT,COUNT_OVERDUE_LOANS,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS


In [25]:
# ANSHIKA

g = df.groupby('SK_ID_CURR')

# DAYS_CREDIT_RECENT
df['DAYS_CREDIT_RECENT'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].max())

# DAYS_CREDIT_OLDEST
df['DAYS_CREDIT_OLDEST'] = df['SK_ID_CURR'].map(g['DAYS_CREDIT'].min())

# AVERAGE_CREDIT_DAY_OVERDUE
df['AVERAGE_CREDIT_DAY_OVERDUE'] = df['SK_ID_CURR'].map(g['CREDIT_DAY_OVERDUE'].mean())

# RECENT_LOAN_COUNT
recent_active = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT'] >= -100)
]

recent_loan_count = recent_active.groupby('SK_ID_CURR').size()
df['RECENT_LOAN_COUNT'] = df['SK_ID_CURR'].map(recent_loan_count).fillna(0).astype(int)

df[df['SK_ID_CURR'] == 100002]


# AVERAGE_DAYS_OVERDUE
# - take average of DAYS_CREDIT_ENDDATE where:
#   CREDIT_ACTIVE == 'Active'
#   DAYS_CREDIT_ENDDATE < 0
# - if no such rows → 0

# Step 1: filter
active_overdue = df[
    (df['CREDIT_ACTIVE'] == 'Active') &
    (df['DAYS_CREDIT_ENDDATE'] < 0)
]

# Step 2: group and average
avg_days_overdue = active_overdue.groupby('SK_ID_CURR')['DAYS_CREDIT_ENDDATE'].mean()

# Step 3: map back and fill missing with 0
df['AVERAGE_DAYS_OVERDUE'] = df['SK_ID_CURR'].map(avg_days_overdue).fillna(0)

df[df['SK_ID_CURR'] == 100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,TOTAL_ACTIVE,TOTAL_CLOSED,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE


In [26]:
g = df.groupby('SK_ID_CURR')

avg_days_credit_update = g['DAYS_CREDIT_UPDATE'].mean()
df['AVERAGE_DAYS_CREDIT_UPDATE'] = df['SK_ID_CURR'].map(avg_days_credit_update)

# AMT_ANNUITY - NORA
# - TOTAL_ANNUITY → sum of active loan's annuities → total monthly obligations

total_annuity = (
    df[df['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR')['AMT_ANNUITY'].sum()
)
df['TOTAL_ANNUITY'] = df['SK_ID_CURR'].map(total_annuity).fillna(0)

df[df['SK_ID_CURR'] == 215354][['SK_ID_CURR', 'AVERAGE_DAYS_CREDIT_UPDATE', 'TOTAL_ANNUITY']]

,SK_ID_CURR,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,215354,-367.272727,0.0
1,215354,-367.272727,0.0
2,215354,-367.272727,0.0
3,215354,-367.272727,0.0
4,215354,-367.272727,0.0
5,215354,-367.272727,0.0
6,215354,-367.272727,0.0
225157,215354,-367.272727,0.0
225158,215354,-367.272727,0.0
225159,215354,-367.272727,0.0


In [27]:
df[df['SK_ID_CURR'] == 100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY


In [28]:
# Make a copy of the original dataframe
df_flat = df.copy()

In [29]:
df_flat = df_flat.drop(columns=['CREDIT_ACTIVE'])
df_flat = df_flat.drop(columns=['CREDIT_CURRENCY'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT'])
df_flat = df_flat.drop(columns=['CREDIT_DAY_OVERDUE'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT_ENDDATE'])
df_flat = df_flat.drop(columns=['DAYS_ENDDATE_FACT'])
df_flat = df_flat.drop(columns=['LOAN_DURATION_SCHEDULED'])
df_flat = df_flat.drop(columns=['LOAN_DURATION_ACTUAL'])
df_flat = df_flat.drop(columns=['LOAN_DEVIATION'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_MAX_OVERDUE'])
df_flat = df_flat.drop(columns=['CNT_CREDIT_PROLONG'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_DEBT'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_LIMIT'])
df_flat = df_flat.drop(columns=['AMT_CREDIT_SUM_OVERDUE'])
df_flat = df_flat.drop(columns=['CREDIT_TYPE'])
df_flat = df_flat.drop(columns=['DAYS_CREDIT_UPDATE'])
df_flat = df_flat.drop(columns=['AMT_ANNUITY'])



In [30]:
df_flat = df_flat.drop(columns=['SK_ID_BUREAU'])

In [31]:
cleaned_application_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
22,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,5.0
56,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
61,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
93,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
103,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307412,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,2.0
307431,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307480,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
307484,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df_flat = df_flat.drop_duplicates(subset='SK_ID_CURR')

In [33]:
# Assuming your application data is called app_df
merged_df = cleaned_application_df.merge(df_flat, on='SK_ID_CURR', how='left')

In [34]:
merged_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0.0,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.5,0.0
3,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20100,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20101,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20102,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
# Check duplicates in the application data itself
merged_df['SK_ID_CURR'].duplicated().sum()

np.int64(0)

In [36]:
df[df['SK_ID_CURR'] == 100002]

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY


In [37]:
df_flat[df_flat['SK_ID_CURR'] == 100002]

,SK_ID_CURR,MAX_AMT_CREDIT_SUM_OVERDUE,DEBT_RATIO,AVERAGE_CREDIT_DAY_OVERDUE,TOTAL_CREDIT_LIMIT,SUM_CNT_CREDIT_PROLONG,CREDIT_TYPE_Another type of loan,CREDIT_TYPE_Car loan,CREDIT_TYPE_Cash loan (non-earmarked),CREDIT_TYPE_Consumer credit,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY


In [38]:
merged_df

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0.0,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.5,0.0
3,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20099,456145,0,Cash loans,F,N,N,0,162000.0,900000.0,29034.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20100,456164,0,Cash loans,F,N,N,1,112500.0,334152.0,18256.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20101,456220,0,Cash loans,F,N,N,1,81000.0,1350000.0,39474.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20102,456228,0,Cash loans,F,Y,N,0,540000.0,545040.0,35617.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [42]:
merged_df_orig = merged_df.copy()
merged_df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,TOTAL_SOLD,TOTAL_BAD_DEBT,COUNT_LOCAL_CURRENCY_LOANS,COUNT_FOREIGN_CURRENCY_LOANS,DAYS_CREDIT_RECENT,DAYS_CREDIT_OLDEST,RECENT_LOAN_COUNT,AVERAGE_DAYS_OVERDUE,AVERAGE_DAYS_CREDIT_UPDATE,TOTAL_ANNUITY
0,100026,0,Cash loans,F,N,N,1,450000.0,497520.0,32521.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100064,0,Cash loans,F,N,N,0,67500.0,298728.0,15381.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100072,0,Cash loans,M,N,N,0,180000.0,1080000.0,44118.0,...,0.0,0.0,2.0,0.0,-494.0,-630.0,0.0,0.0,-214.5,0.0
3,100111,0,Cash loans,F,Y,N,1,112500.0,862560.0,27954.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100122,0,Cash loans,F,N,N,1,76500.0,808650.0,26217.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20104 entries, 0 to 20103
Columns: 154 entries, SK_ID_CURR to TOTAL_ANNUITY
dtypes: float64(97), int64(41), object(16)
memory usage: 23.6+ MB


In [44]:
merged_df.select_dtypes('number').max().map('{:,.0f}'.format)

SK_ID_CURR                      456,251
TARGET                                1
CNT_CHILDREN                          9
AMT_INCOME_TOTAL              3,600,000
AMT_CREDIT                    3,375,000
                                ...    
DAYS_CREDIT_OLDEST                   -3
RECENT_LOAN_COUNT                     3
AVERAGE_DAYS_OVERDUE                  0
AVERAGE_DAYS_CREDIT_UPDATE           -1
TOTAL_ANNUITY                   823,036
Length: 138, dtype: object

In [ ]:
merged_df.columns.tolist()


0        N
1        N
2        N
3        N
4        N
        ..
20099    N
20100    N
20101    N
20102    N
20103    N
Name: FLAG_OWN_REALTY, Length: 20104, dtype: object

In [50]:
print(merged_df['TARGET'].value_counts())
print(merged_df['TARGET'].unique())

TARGET
0    17984
1     2120
Name: count, dtype: int64
[0 1]


In [54]:
# Select only numeric columns for correlation
numeric_df = merged_df.select_dtypes(include='number')

# Compute correlation
corr = numeric_df.corr(method='pearson')

# View correlation with TARGET
print(corr['TARGET'].sort_values(ascending=False))

TARGET                                            1.000000
DEBT_RATIO                                        0.082637
REGION_RATING_CLIENT                              0.078608
REGION_RATING_CLIENT_W_CITY                       0.077932
FLAG_DOCUMENT_3                                   0.069213
                                                    ...   
CREDIT_TYPE_Cash loan (non-earmarked)                  NaN
CREDIT_TYPE_Loan for the purchase of equipment         NaN
CREDIT_TYPE_Mobile operator loan                       NaN
CREDIT_TYPE_Real estate loan                           NaN
TOTAL_BAD_DEBT                                         NaN
Name: TARGET, Length: 138, dtype: float64


In [55]:
# See which columns have NaN correlation with TARGET
nan_corr = corr['TARGET'][corr['TARGET'].isna()]
print("Columns with NaN correlation:\n", nan_corr.index.tolist())

# Drop them from merged_df
merged_df = merged_df.drop(columns=nan_corr.index.tolist(), errors='ignore')

Columns with NaN correlation:
 ['FLAG_MOBIL', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_12', 'CREDIT_TYPE_Cash loan (non-earmarked)', 'CREDIT_TYPE_Loan for the purchase of equipment', 'CREDIT_TYPE_Mobile operator loan', 'CREDIT_TYPE_Real estate loan', 'TOTAL_BAD_DEBT']


In [57]:
# Select only numeric columns for correlation
numeric_df = merged_df.select_dtypes(include='number')

# Compute correlation
corr = numeric_df.corr(method='pearson')

# View correlation with TARGET
print(corr['TARGET'].sort_values(ascending=False))

pd.set_option('display.max_rows', None)
print(corr['TARGET'].sort_values(ascending=False))

TARGET                         1.000000
DEBT_RATIO                     0.082637
REGION_RATING_CLIENT           0.078608
REGION_RATING_CLIENT_W_CITY    0.077932
FLAG_DOCUMENT_3                0.069213
                                 ...   
ELEVATORS_MODE                -0.044889
ELEVATORS_MEDI                -0.045739
ELEVATORS_AVG                 -0.046999
REGION_POPULATION_RELATIVE    -0.048478
EXT_SOURCE_1                  -0.137411
Name: TARGET, Length: 129, dtype: float64
TARGET                                                1.000000
DEBT_RATIO                                            0.082637
REGION_RATING_CLIENT                                  0.078608
REGION_RATING_CLIENT_W_CITY                           0.077932
FLAG_DOCUMENT_3                                       0.069213
MAX_AMT_CREDIT_SUM_OVERDUE                            0.068188
REG_CITY_NOT_WORK_CITY                                0.062381
DAYS_CREDIT_OLDEST                                    0.061384
RECENT_LOAN_CO

In [58]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Drop TARGET and any non-numeric columns
X = merged_df.select_dtypes(include='number').drop(columns=['TARGET'], errors='ignore')
y = merged_df['TARGET']

# Handle missing values (logistic regression can't handle NaNs)
X = X.fillna(X.median())

# Create and fit logistic regression model
logreg = LogisticRegression(solver='liblinear', class_weight='balanced', max_iter=1000)
logreg.fit(X, y)

# View output coefficients
coeff = np.concatenate([logreg.intercept_, logreg.coef_.reshape(-1)])
pd.Series(coeff, index=['Intercept'] + X.columns.tolist()).sort_values(ascending=False)

REGION_RATING_CLIENT                                  5.546505e-02
REGION_RATING_CLIENT_W_CITY                           5.477102e-02
FLAG_DOCUMENT_3                                       3.899739e-02
REG_CITY_NOT_WORK_CITY                                3.214443e-02
AMT_REQ_CREDIT_BUREAU_YEAR                            3.026105e-02
COUNT_LOCAL_CURRENCY_LOANS                            2.776887e-02
FLAG_WORK_PHONE                                       2.386264e-02
REG_CITY_NOT_LIVE_CITY                                2.208335e-02
TOTAL_ACTIVE                                          2.076441e-02
DEF_30_CNT_SOCIAL_CIRCLE                              1.807659e-02
LIVE_CITY_NOT_WORK_CITY                               1.626587e-02
DEF_60_CNT_SOCIAL_CIRCLE                              1.509567e-02
CREDIT_TYPE_Consumer credit                           1.302763e-02
CREDIT_TYPE_Credit card                               1.258090e-02
CNT_FAM_MEMBERS                                       8.615510

In [59]:
# prepare data - all numeric features except TARGET
X = merged_df.select_dtypes(include='number').drop(columns=['TARGET'], errors='ignore')
y = merged_df['TARGET']

# handle missing values
X = X.fillna(X.median())

# fit logistic regression on ALL features
logreg = LogisticRegression(solver='liblinear', class_weight='balanced', max_iter=1000)
logreg.fit(X, y)

# concatenate intercept and coefficients
coeff = np.concatenate([logreg.intercept_, logreg.coef_.reshape(-1)])

# create series with feature names
coeff_series = pd.Series(coeff, index=['Intercept'] + X.columns.tolist())

# sort so most influential features are easy to spot
coeff_series.sort_values(ascending=False)

REGION_RATING_CLIENT                                  5.546505e-02
REGION_RATING_CLIENT_W_CITY                           5.477102e-02
FLAG_DOCUMENT_3                                       3.899739e-02
REG_CITY_NOT_WORK_CITY                                3.214443e-02
AMT_REQ_CREDIT_BUREAU_YEAR                            3.026105e-02
COUNT_LOCAL_CURRENCY_LOANS                            2.776887e-02
FLAG_WORK_PHONE                                       2.386264e-02
REG_CITY_NOT_LIVE_CITY                                2.208335e-02
TOTAL_ACTIVE                                          2.076441e-02
DEF_30_CNT_SOCIAL_CIRCLE                              1.807659e-02
LIVE_CITY_NOT_WORK_CITY                               1.626587e-02
DEF_60_CNT_SOCIAL_CIRCLE                              1.509567e-02
CREDIT_TYPE_Consumer credit                           1.302763e-02
CREDIT_TYPE_Credit card                               1.258090e-02
CNT_FAM_MEMBERS                                       8.615510

In [62]:
sample_row = X.iloc[0]
data = sample_row.to_dict()

# compute linear combination starting with intercept
linear_combination = coeff_series['Intercept']

# add each feature's contribution
for key in data.keys():
    if key in coeff_series.index:  # safety check in case a key is missing
        linear_combination += coeff_series[key] * data[key]

# compute probability using logistic regression formula
probability = 1 / (1 + np.exp(-linear_combination))
pred_status = int(probability > 0.5)

print(f'Probability of default = {probability:.3f}')
print(f'Predicted status = {pred_status}  (1 = default, 0 = repaid)')


Probability of default = 0.500
Predicted status = 1  (1 = default, 0 = repaid)


In [63]:
from sklearn.metrics import accuracy_score

# generate predictions for all rows
y_pred = logreg.predict(X)

# calculate accuracy
accuracy = accuracy_score(y, y_pred)
print(f'Model accuracy: {accuracy:.4f}')

Model accuracy: 0.5696


In [66]:
def woe_iv(data):
    working_data = data.copy()

    # handle missing values from qcut
    working_data['Bin_Range'] = working_data['Bin_Range'].astype('object')
    working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')

    variable_data = pd.DataFrame()
    variable_data['Bin_Range'] = working_data.groupby(by='Bin_Range', as_index=False).count()['Bin_Range']
    variable_data['Count'] = working_data.groupby(by='Bin_Range', as_index=False).count()['TARGET']
    variable_data['Events'] = working_data.groupby(by='Bin_Range', as_index=False).sum()['TARGET']
    variable_data['Non_Events'] = variable_data['Count'] - variable_data['Events']
    variable_data['%_of_Events'] = variable_data['Events'] / sum(variable_data['Events'])
    variable_data['%_of_Non_Events'] = variable_data['Non_Events'] / sum(variable_data['Non_Events'])
    variable_data['WOE'] = np.log(variable_data['%_of_Non_Events'] / variable_data['%_of_Events'])
    variable_data['IV'] = (variable_data['%_of_Non_Events'] - variable_data['%_of_Events']) * variable_data['WOE']
    variable_data['total_IV'] = variable_data['IV'].sum()

    return variable_data

# test the function on your strongest predictor
woe_bin_data = merged_df[['EXT_SOURCE_1', 'TARGET']].copy()
woe_bin_data['Bin_Range'] = pd.qcut(merged_df['EXT_SOURCE_1'], q=10, duplicates='drop')
woe_iv(woe_bin_data)

,Bin_Range,Count,Events,Non_Events,%_of_Events,%_of_Non_Events,WOE,IV,total_IV
0,"(0.013600000000000001, 0.161]",1014,194,820,0.091509,0.045596,-0.696620,0.031984,0.116769
1,"(0.161, 0.223]",1014,148,866,0.069811,0.048154,-0.371394,0.008043,0.116769
2,"(0.223, 0.278]",1014,138,876,0.065094,0.048710,-0.289954,0.004751,0.116769
3,"(0.278, 0.335]",1013,113,900,0.053302,0.050044,-0.063059,0.000205,0.116769
4,"(0.335, 0.394]",1014,100,914,0.047170,0.050823,0.074594,0.000273,0.116769
5,"(0.394, 0.456]",1014,79,935,0.037264,0.051991,0.333032,0.004904,0.116769
6,"(0.456, 0.531]",1013,77,936,0.036321,0.052046,0.359744,0.005657,0.116769
7,"(0.531, 0.613]",1014,55,959,0.025943,0.053325,0.720492,0.019728,0.116769
8,"(0.613, 0.715]",1014,77,937,0.036321,0.052102,0.360811,0.005694,0.116769
9,"(0.715, 0.926]",1014,42,972,0.019811,0.054048,1.003620,0.034361,0.116769


In [67]:
screening_vars = [
    'EXT_SOURCE_1', 'DEBT_RATIO', 'REGION_RATING_CLIENT',
    'MAX_AMT_CREDIT_SUM_OVERDUE', 'DAYS_CREDIT_OLDEST',
    'DAYS_BIRTH', 'AMT_CREDIT', 'AMT_GOODS_PRICE',
    'DAYS_EMPLOYED', 'ELEVATORS_AVG'
]

results = []

for var in screening_vars:
    woe_bin_data = merged_df[[var, 'TARGET']].copy()
    woe_bin_data['Bin_Range'] = pd.qcut(merged_df[var], q=20, duplicates='drop')
    output = woe_iv(woe_bin_data)
    total_iv = output['total_IV'].iloc[0]
    results.append([var, round(total_iv, 4)])

# step 3: view IV table and decide what to drop
iv_table = pd.DataFrame(results, columns=['Variable', 'IV'])
iv_table = iv_table.sort_values('IV', ascending=False).reset_index(drop=True)
iv_table

/var/folders/5_/5bgkd51n45sgbnmt71swhhsw0000gn/T/ipykernel_29775/1733306195.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
/var/folders/5_/5bgkd51n45sgbnmt71swhhsw0000gn/T/ipykernel_29775/1733306195.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  working_data['Bin_Range'] = working_data['Bin_Range'].fillna('Missing')
/var/folders/5_/5bgkd51n45sgbnmt71swhhsw0000gn/T/ipykernel_29775/1733306195.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is d

,Variable,IV
0,EXT_SOURCE_1,0.1236
1,AMT_GOODS_PRICE,0.1115
2,AMT_CREDIT,0.0811
3,DAYS_EMPLOYED,0.0725
4,DAYS_BIRTH,0.0578
5,ELEVATORS_AVG,0.0442
6,REGION_RATING_CLIENT,0.0337
7,DEBT_RATIO,0.0261
8,DAYS_CREDIT_OLDEST,0.0191
9,MAX_AMT_CREDIT_SUM_OVERDUE,0.0024


In [68]:
weak_vars = iv_table[iv_table['IV'] < 0.02]['Variable'].tolist()
print("Dropping:", weak_vars)
merged_df = merged_df.drop(columns=weak_vars, errors='ignore')

Dropping: ['DAYS_CREDIT_OLDEST', 'MAX_AMT_CREDIT_SUM_OVERDUE']
